# SC Dataset Exploration

This notebook explores the South Carolina (SC) dataset to understand household counts, income distribution, and demographic characteristics.

In [1]:
from policyengine_us import Microsimulation
import pandas as pd
import numpy as np

SC_DATASET = "hf://policyengine/policyengine-us-data/states/SC.h5"

In [2]:
# Load SC dataset
sim = Microsimulation(dataset=SC_DATASET)

In [ ]:
# Check dataset size - use .values to get raw arrays (avoid MicroSeries auto-weighting)
household_weight = sim.calculate("household_weight", period=2025).values
household_count = sim.calculate("household_count", period=2025, map_to="household").values
person_count = sim.calculate("person_count", period=2025, map_to="household").values

# Weighted sums using raw arrays
weighted_household_count = (household_count * household_weight).sum()
weighted_person_count = (person_count * household_weight).sum()

print(f"Number of households in dataset: {len(household_weight):,}")
print(f"Household count (weighted): {weighted_household_count:,.0f}")
print(f"Person count (weighted): {weighted_person_count:,.0f}")

In [ ]:
# Check income distribution (weighted vs unweighted, household and person level)
# Use .values to get raw numpy arrays (avoid MicroSeries auto-weighting)
agi_hh_array = sim.calculate("adjusted_gross_income", period=2025, map_to="household").values
hh_weights = sim.calculate("household_weight", period=2025).values

agi_person_array = sim.calculate("adjusted_gross_income", period=2025, map_to="person").values
person_weights = sim.calculate("person_weight", period=2025).values

# Weighted percentile calculation
def weighted_percentile(values, weights, percentile):
    sorted_indices = np.argsort(values)
    sorted_values = values[sorted_indices]
    sorted_weights = weights[sorted_indices]
    cumulative_weight = np.cumsum(sorted_weights)
    idx = np.searchsorted(cumulative_weight, cumulative_weight[-1] * percentile / 100)
    return sorted_values[min(idx, len(sorted_values)-1)]

# Unweighted medians
unweighted_median_hh = np.median(agi_hh_array)
unweighted_median_person = np.median(agi_person_array)

# Weighted medians
weighted_median_hh = weighted_percentile(agi_hh_array, hh_weights, 50)
weighted_median_person = weighted_percentile(agi_person_array, person_weights, 50)

# Weighted averages
weighted_avg_hh = np.average(agi_hh_array, weights=hh_weights)
weighted_avg_person = np.average(agi_person_array, weights=person_weights)

# Average household size
total_persons = person_weights.sum()
total_households = hh_weights.sum()
avg_hh_size = total_persons / total_households

print("=" * 60)
print("INCOME DISTRIBUTION SUMMARY")
print("=" * 60)
print(f"\nHousehold AGI:")
print(f"  Unweighted median: ${unweighted_median_hh:,.0f}")
print(f"  Weighted median:   ${weighted_median_hh:,.0f}")
print(f"  Weighted average:  ${weighted_avg_hh:,.0f}")

print(f"\nPerson AGI:")
print(f"  Unweighted median: ${unweighted_median_person:,.0f}")
print(f"  Weighted median:   ${weighted_median_person:,.0f}")
print(f"  Weighted average:  ${weighted_avg_person:,.0f}")

print(f"\nAverage household size: {avg_hh_size:.1f}")

print(f"\nWeighted household AGI percentiles:")
print(f"  25th percentile: ${weighted_percentile(agi_hh_array, hh_weights, 25):,.0f}")
print(f"  50th percentile: ${weighted_percentile(agi_hh_array, hh_weights, 50):,.0f}")
print(f"  75th percentile: ${weighted_percentile(agi_hh_array, hh_weights, 75):,.0f}")
print(f"  90th percentile: ${weighted_percentile(agi_hh_array, hh_weights, 90):,.0f}")
print(f"  95th percentile: ${weighted_percentile(agi_hh_array, hh_weights, 95):,.0f}")
print(f"  Max AGI: ${agi_hh_array.max():,.0f}")

In [ ]:
# Check households with children - use .values for raw arrays
is_child = sim.calculate("is_child", period=2025, map_to="person").values
household_id = sim.calculate("household_id", period=2025, map_to="person").values
household_weight_person = sim.calculate("household_weight", period=2025, map_to="person").values

# Create DataFrame
df_households = pd.DataFrame({
    'household_id': household_id,
    'is_child': is_child,
    'household_weight': household_weight_person
})

# Count children per household
children_per_household = df_households.groupby('household_id').agg({
    'is_child': 'sum',
    'household_weight': 'first'
}).reset_index()

# Calculate weighted household counts
total_households_with_children = children_per_household[children_per_household['is_child'] > 0]['household_weight'].sum()
households_with_1_child = children_per_household[children_per_household['is_child'] == 1]['household_weight'].sum()
households_with_2_children = children_per_household[children_per_household['is_child'] == 2]['household_weight'].sum()
households_with_3plus_children = children_per_household[children_per_household['is_child'] >= 3]['household_weight'].sum()

print(f"\nHouseholds with children (weighted):")
print(f"  Total households with children: {total_households_with_children:,.0f}")
print(f"  Households with 1 child: {households_with_1_child:,.0f}")
print(f"  Households with 2 children: {households_with_2_children:,.0f}")
print(f"  Households with 3+ children: {households_with_3plus_children:,.0f}")

In [ ]:
# Check children by age groups - use .values for raw arrays
df = pd.DataFrame({
    "household_id": sim.calculate("household_id", map_to="person").values,
    "tax_unit_id": sim.calculate("tax_unit_id", map_to="person").values,
    "person_id": sim.calculate("person_id", map_to="person").values,
    "age": sim.calculate("age", map_to="person").values,
    "person_weight": sim.calculate("person_weight", map_to="person").values
})

# Filter for children and apply weights
children_under_18_df = df[df['age'] < 18]
children_under_6_df = df[df['age'] < 6]
children_under_3_df = df[df['age'] < 3]

# Calculate weighted totals
total_children = children_under_18_df['person_weight'].sum()
children_under_6 = children_under_6_df['person_weight'].sum()
children_under_3 = children_under_3_df['person_weight'].sum()

print(f"\nChildren by age:")
print(f"  Total children under 18: {total_children:,.0f}")
print(f"  Children under 6: {children_under_6:,.0f}")
print(f"  Children under 3: {children_under_3:,.0f}")

In [ ]:
# Create comprehensive summary table
summary_data = {
    'Metric': [
        'Household count (weighted)',
        'Person count (weighted)',
        'Average household size',
        'Weighted median household AGI',
        'Weighted average household AGI',
        'Weighted median person AGI',
        'Weighted average person AGI',
        'Unweighted median household AGI',
        'Unweighted median person AGI',
        '25th percentile household AGI',
        '75th percentile household AGI',
        '90th percentile household AGI',
        '95th percentile household AGI',
        'Max household AGI',
        'Total households with children',
        'Households with 1 child',
        'Households with 2 children',
        'Households with 3+ children',
        'Total children under 18',
        'Children under 6',
        'Children under 3'
    ],
    'Value': [
        f"{weighted_household_count:,.0f}",
        f"{weighted_person_count:,.0f}",
        f"{avg_hh_size:.1f}",
        f"${weighted_median_hh:,.0f}",
        f"${weighted_avg_hh:,.0f}",
        f"${weighted_median_person:,.0f}",
        f"${weighted_avg_person:,.0f}",
        f"${unweighted_median_hh:,.0f}",
        f"${unweighted_median_person:,.0f}",
        f"${weighted_percentile(agi_hh_array, hh_weights, 25):,.0f}",
        f"${weighted_percentile(agi_hh_array, hh_weights, 75):,.0f}",
        f"${weighted_percentile(agi_hh_array, hh_weights, 90):,.0f}",
        f"${weighted_percentile(agi_hh_array, hh_weights, 95):,.0f}",
        f"${agi_hh_array.max():,.0f}",
        f"{total_households_with_children:,.0f}",
        f"{households_with_1_child:,.0f}",
        f"{households_with_2_children:,.0f}",
        f"{households_with_3plus_children:,.0f}",
        f"{total_children:,.0f}",
        f"{children_under_6:,.0f}",
        f"{children_under_3:,.0f}"
    ]
}

summary_df = pd.DataFrame(summary_data)

print("\n" + "="*65)
print("SC DATASET SUMMARY - WEIGHTED (Population Estimates)")
print("="*65)
print(summary_df.to_string(index=False))
print("="*65)

# Save table
summary_df.to_csv('sc_dataset_summary_weighted.csv', index=False)
print("\nSummary saved to: sc_dataset_summary_weighted.csv")

In [ ]:
# Households with $0 income - using raw arrays
agi_hh = sim.calculate("adjusted_gross_income", period=2025, map_to="household").values
weights = sim.calculate("household_weight", period=2025).values

zero_income_mask = agi_hh == 0
zero_income_count = weights[zero_income_mask].sum()
total_households = weights.sum()

print("\n" + "="*70)
print("HOUSEHOLDS WITH $0 INCOME")
print("="*70)
print(f"Household count: {zero_income_count:,.0f}")
print(f"Percentage of all households: {zero_income_count / total_households * 100:.2f}%")
print("="*70)

In [9]:
# Household counts by income brackets
income_brackets = [
    (0, 10000, "$0-$10k"),
    (10000, 20000, "$10k-$20k"),
    (20000, 30000, "$20k-$30k"),
    (30000, 40000, "$30k-$40k"),
    (40000, 50000, "$40k-$50k"),
    (50000, 60000, "$50k-$60k")
]

bracket_data = []
for lower, upper, label in income_brackets:
    mask = (agi_hh >= lower) & (agi_hh < upper)
    count = weights[mask].sum()
    pct_of_total = (count / total_households) * 100
    
    bracket_data.append({
        "Income Bracket": label,
        "Households": f"{count:,.0f}",
        "% of All Households": f"{pct_of_total:.2f}%"
    })

income_df = pd.DataFrame(bracket_data)

print("\n" + "="*70)
print("HOUSEHOLD COUNTS BY INCOME BRACKET")
print("="*70)
print(income_df.to_string(index=False))
print("="*70)

# Total in $0-$60k range
total_in_range = sum([weights[(agi_hh >= lower) & (agi_hh < upper)].sum() for lower, upper, _ in income_brackets])
print(f"\nTotal households in $0-$60k range: {total_in_range:,.0f}")
print(f"Percentage of all households in $0-$60k range: {total_in_range / total_households * 100:.2f}%")


HOUSEHOLD COUNTS BY INCOME BRACKET
Income Bracket Households % of All Households
       $0-$10k    434,505              23.02%
     $10k-$20k    155,370               8.23%
     $20k-$30k    149,595               7.93%
     $30k-$40k    115,365               6.11%
     $40k-$50k    127,566               6.76%
     $50k-$60k    110,405               5.85%

Total households in $0-$60k range: 1,092,805
Percentage of all households in $0-$60k range: 57.90%
